In [1]:
import torch.nn as nn
import torch
import pandas as pd
import copy
import random
import numpy as np

In [2]:
max_len = 21
embed_dim = 256


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
set_seed()

In [4]:
df = pd.read_csv('/Users/baonguyen/IU/thesis/data/clean_data/data_with_bertopic_column.csv')


In [5]:
df['review_date']=pd.to_datetime(df['review_date'])
df_sorted = df.sort_values('review_date')

In [6]:
unique_item_id = set(df_sorted['item_id'])
item_to_index = {item:idx +1 for idx , item in enumerate(unique_item_id)}
index_to_item = {idx+1:item for idx , item in enumerate(unique_item_id)}

In [7]:
# Step 1: Group and aggregate
user_item_sequence = (
    df_sorted.groupby('user_id')[['item_id']]
    .agg(list)
    .to_dict(orient='index')
)

# Step 2: Remove users with fewer than 2 item_ids
user_item_sequence = {
    user: val
    for user, val in user_item_sequence.items()
    if len(val['item_id']) >= 2 and len(val['item_id'])<=21
}


In [8]:
user_item_to_index_sequence = {}
for user,value in user_item_sequence.items():
    user_item_to_index_sequence[user] = {'item_id':[item_to_index[item] for item in value['item_id']]}

In [9]:
def mask_sequence(sequence: dict, mask_ratio: float):
    labels = {}
    mask_seq = {}
    for user, seq in sequence.items():
        mask_seq[user] = copy.deepcopy(seq)  # Deep copy so original is untouched
        labels[user] = [-100] * len(seq['item_id'])
        for i in range(len(mask_seq[user]['item_id'])):
            if random.random() < mask_ratio:
                labels[user][i] = mask_seq[user]['item_id'][i]  # Save original item id
                mask_seq[user]['item_id'][i] = 0       # Mask the item id
               
    return mask_seq, labels


In [10]:
def padding(mask_seq, labels, max_len=64, pad_item=0, pad_topic=0, pad_label=-100):
    """
    Pads all user sequences in mask_seq and labels to max_len.
    
    Args:
        mask_seq: dict of user_id -> {'item_id': [...], 'Topic': [...]}
        labels: dict of user_id -> [...]
        max_len: desired length after padding
        pad_item: value for padding 'item_id'
        pad_topic: value for padding 'Topic'
        pad_label: value for padding labels

    Returns:
        padded_mask_seq, padded_labels (dicts)
    """
    def pad(seq, max_len, pad_value):
        if len(seq) < max_len:
            return seq + [pad_value] * (max_len - len(seq))
        else:
            return seq[len(seq)-max_len:len(seq)]
    
    padded_mask_seq = {}
    padded_labels = {}

    for user in mask_seq:
        padded_mask_seq[user] = {
            'item_id': pad(mask_seq[user]['item_id'], max_len, pad_item)
        }
        padded_labels[user] = pad(labels[user], max_len, pad_label)
    
    return padded_mask_seq, padded_labels


# bert architect


In [11]:
class BertEmbeddings(nn.Module):
    def __init__(self,vocab_size, hidden_size, max_len, dropout):
        super().__init__()
        self.max_len = max_len
        self.word_embeddings = nn.Embedding(vocab_size,hidden_size)
        self.position_encoding = nn.Embedding(max_len,hidden_size)
        self.LayerNorm = nn.LayerNorm(hidden_size)
        self.Dropout = nn.Dropout(dropout)

    def forward(self,input_ids):
        position_ids = torch.arange(self.max_len, dtype=torch.long, device=input_ids.device).unsqueeze(0)
        word_emb = self.word_embeddings(input_ids)
        pos_emb = self.position_encoding(position_ids)
        embeddings = word_emb + pos_emb
        embeddings = self.LayerNorm(embeddings)
        return self.Dropout(embeddings)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class BertSdpaSelfAttention(nn.Module):
    def __init__(self, hidden_size=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None,key_padding_mask=None):
        B, T, C = x.size()

        # Linear projection and reshape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            scores += attention_mask
        if key_padding_mask is not None:
            key_padding_mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(key_padding_mask,float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        attn_weights = self.attn_dropout(attn_weights)

        context = torch.matmul(attn_weights, v)  # [B, H, T, D]
        context = context.transpose(1, 2).reshape(B, T, C)
        return context

class BertSelfOutput(nn.Module):
    def __init__(self, hidden_size=512, dropout=0.1):
        super().__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.LayerNorm = nn.LayerNorm(hidden_size)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return self.LayerNorm(hidden_states + input_tensor)

class BertAttention(nn.Module):
    def __init__(self, hidden_size=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.self = BertSdpaSelfAttention(hidden_size, num_heads, dropout)
        self.output = BertSelfOutput(hidden_size, dropout)

    def forward(self, hidden_states, attention_mask=None,key_padding_mask=None):
        self_output = self.self(hidden_states, attention_mask,key_padding_mask)
        return self.output(self_output, hidden_states)

class BertIntermediate(nn.Module):
    def __init__(self, hidden_size=512, intermediate_size=3072):
        super().__init__()
        self.dense = nn.Linear(hidden_size, intermediate_size)
        self.activation = nn.GELU()

    def forward(self, hidden_states):
        return self.activation(self.dense(hidden_states))

class BertOutput(nn.Module):
    def __init__(self, intermediate_size=3072, hidden_size=512, dropout=0.1):
        super().__init__()
        self.dense = nn.Linear(intermediate_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.LayerNorm = nn.LayerNorm(hidden_size)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return self.LayerNorm(hidden_states + input_tensor)

class BertLayer(nn.Module):
    def __init__(self, hidden_size=512, intermediate_size=3072, num_heads=8, dropout=0.1):
        super().__init__()
        self.attention = BertAttention(hidden_size, num_heads, dropout)
        self.intermediate = BertIntermediate(hidden_size, intermediate_size)
        self.output = BertOutput(intermediate_size, hidden_size, dropout)

    def forward(self, hidden_states, attention_mask=None,key_padding_mask=None):
        attention_output = self.attention(hidden_states, attention_mask,key_padding_mask)
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output


In [13]:
import torch
import torch.nn as nn

class BertEncoder(nn.Module):
    def __init__(self, num_layers=2, hidden_size=512, intermediate_size=3072, num_heads=8, dropout=0.1):
        super().__init__()
        self.layer = nn.ModuleList([
            BertLayer(hidden_size, intermediate_size, num_heads, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, hidden_states, attention_mask=None,key_padding_mask=None):
        for layer_module in self.layer:
            hidden_states = layer_module(hidden_states, attention_mask,key_padding_mask)
        return hidden_states


In [14]:
import torch
import torch.nn as nn



class BertModel(nn.Module):
    def __init__(self, 
                 vocab_size=30522,
                 hidden_size=512,
                 intermediate_size=3072,
                 num_heads=8,
                 num_layers=2,
                 max_len=512,
                 dropout=0.1):
        super().__init__()
        self.embeddings = BertEmbeddings(vocab_size, hidden_size, max_len, dropout=dropout)
        self.encoder = BertEncoder(num_layers, hidden_size, intermediate_size, num_heads, dropout)
        self.output_layer = nn.Sequential(
           nn.Dropout(dropout),
            nn.Linear(hidden_size, vocab_size)
        )

    def forward(self, input_ids, attention_mask=None,key_padding_mask=None):
        embedding_output = self.embeddings(input_ids)
        encoder_output = self.encoder(embedding_output, attention_mask,key_padding_mask)
        output = self.output_layer(encoder_output)
        return output


In [15]:
from torchinfo import summary
model = BertModel()
summary(model,depth=6)

Layer (type:depth-idx)                                  Param #
BertModel                                               --
├─BertEmbeddings: 1-1                                   --
│    └─Embedding: 2-1                                   15,627,264
│    └─Embedding: 2-2                                   262,144
│    └─LayerNorm: 2-3                                   1,024
│    └─Dropout: 2-4                                     --
├─BertEncoder: 1-2                                      --
│    └─ModuleList: 2-5                                  --
│    │    └─BertLayer: 3-1                              --
│    │    │    └─BertAttention: 4-1                     --
│    │    │    │    └─BertSdpaSelfAttention: 5-1        --
│    │    │    │    │    └─Linear: 6-1                  262,656
│    │    │    │    │    └─Linear: 6-2                  262,656
│    │    │    │    │    └─Linear: 6-3                  262,656
│    │    │    │    │    └─Dropout: 6-4                 --
│    │    │    │    

In [16]:
def precision_at_k(ground_truth: list, prediction: list, k: int):
    precisions = []
    for gt_item, pred in zip(ground_truth, prediction):
        recommended = pred[:k]
        hit = 1 if gt_item in recommended else 0
        precisions.append(hit / k)
    return sum(precisions) / len(precisions)

def recall_at_k(ground_truth: list, prediction: list, k: int):
    recalls = []
    for gt_item, pred in zip(ground_truth, prediction):
        recommended = pred[:k]
        hit = 1 if gt_item in recommended else 0
        recalls.append(hit)
    return sum(recalls) / len(recalls)

def mrr(ground_truth: list, prediction: list):
    rr = []
    for gt_item, pred in zip(ground_truth, prediction):
        if gt_item in pred:
            rank = pred.index(gt_item) + 1
            rr.append(1.0 / rank)
        else:
            rr.append(0.0)
    return sum(rr) / len(rr)

import math

def ndcg_at_k(ground_truth: list, prediction: list, k: int):
    ndcgs = []
    for gt_item, pred in zip(ground_truth, prediction):
        if gt_item in pred[:k]:
            rank = pred.index(gt_item) + 1
            dcg = 1 / math.log2(rank + 1)
            idcg = 1.0  # since only one ground truth item
            ndcgs.append(dcg / idcg)
        else:
            ndcgs.append(0.0)
    return sum(ndcgs) / len(ndcgs)

def coverage(prediction: list, catalog: set):
    recommended_items = set(item for user_pred in prediction for item in user_pred)
    return len(recommended_items) / len(catalog)

In [17]:
def hit_ratio(ground_truth:list,prediction:list,k:int):
    hits = 0
    total = len(ground_truth)
    for i, (gt_item, pred) in enumerate(zip(ground_truth, prediction)):
        
        if gt_item in pred[:k]:
            print(f"[Sample {i}] GT: {gt_item}, Pred top-{k}: {pred[:k]}")
            hits += 1
    return hits / max(1,total)

# -------------------------
# Function to load the popularity data (counts.csv)
def load_popularity_data(filepath):
    df = pd.read_csv(filepath)
    item_popularity = dict(zip(df['item_id'], df['count']))  # Item popularity dictionary
    total_count = sum(item_popularity.values())  # Total count of interactions
    item_probabilities = {item: count / total_count for item, count in item_popularity.items()}  # Normalize probabilities
    return item_popularity, item_probabilities
# -------------------------
# Function to sample negative items based on popularity
def sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=None):
    """Sample N negative items based on popularity, excluding the ground truth."""
    possible_negatives = all_items - set(interacted_item)
    negatives = np.random.choice(
    a=list(possible_negatives),                                    # candidates
    size=min(num_negatives, len(possible_negatives)),              # sample size
    replace=False,                                                 # no duplicates
    p=np.array([item_probabilities.get(item, 0) 
                for item in possible_negatives], dtype=float) / 
      max(1e-12, sum(item_probabilities.get(item, 0) 
                     for item in possible_negatives))              # normalize weights
).tolist()
    
    return negatives
# -------------------------
item_popularity, item_probabilities = load_popularity_data('/Users/baonguyen/IU/thesis/data/counts.csv')
item_probabilities = {item_to_index[key]:value for key,value in item_probabilities.items()}
all_items = [i for i in range(len(unique_item_id)+1)]
all_items=set(all_items)
def evaluate_model(model, val_item_sequences, k=10):
    model.eval()
    device = 'mps'

    ground_truths = []
    predictions = []

    with torch.no_grad():
        for user, seq in val_item_sequences.items():
            item_seq = seq['item_id']
            


            # Prepare input and target
            input_items = item_seq[:-1]
            
            target_item = item_seq[-1]

            # Use your own padding utility to ensure correct length
            padded_seq, _ = padding(
                mask_seq={user: {'item_id': input_items}},
                labels={user: []},  # empty labels not needed here
                max_len=max_len
            )
            
            padded_items = padded_seq[user]['item_id']
            # padded_topics = padded_seq[user]['Topic']

            item_tensor = torch.tensor([padded_items], dtype=torch.long).to(device)
            
            key_padding_mask = (item_tensor == 0)

            logits = model(item_tensor, key_padding_mask=key_padding_mask)[:,min(len(item_seq)-1,max_len-1),:]
            probabilities = torch.softmax(logits, dim=-1)
            # --- Popularity-based Negative Sampling ---
            # Sample N negative items (those not interacted with by the user)
            negatives = sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=item_seq)
            candidates = [target_item] + negatives

            # Get probabilities for the candidate items only
            candidate_logits = probabilities[0, candidates]  # Shape: (N+1,)
            
            # Rank candidates by their logits (probabilities)
            ranked = [x for _, x in sorted(zip(candidate_logits.tolist(), candidates), reverse=True)]

            # Store the ground truth and top-k predictions
            ground_truths.append(index_to_item[target_item])
            predictions.append([index_to_item[i] for i in ranked])

            # print(ground_truths)
            # print(predictions)
    return hit_ratio(ground_truths, predictions, k), \
            precision_at_k(ground_truths, predictions, k), \
            recall_at_k(ground_truths, predictions, k), \
              mrr(ground_truths, predictions), \
              ndcg_at_k(ground_truths, predictions, k), \
              coverage(predictions, set(index_to_item.values()))



In [18]:
import torch
import torch.optim as optim
from tqdm import tqdm
import os

epoch_num = 10
hitrate = 5

def train_model(train_users, val_users, fold_num, mask_ratio):
    # 1) Build training tensors
    train_user_item_to_index_sequence = {
        user: seq for user, seq in user_item_to_index_sequence.items() if user in train_users
    }
    mask_seq, labels = mask_sequence(train_user_item_to_index_sequence, mask_ratio=mask_ratio)
    padded_mask_seq, padded_labels = padding(mask_seq, labels, max_len=max_len)

    tensor_item_ids = torch.stack([torch.tensor(u['item_id']) for u in padded_mask_seq.values()])
    tensor_labels   = torch.stack([torch.tensor(seq) for seq in padded_labels.values()])

    train_dataset = torch.utils.data.TensorDataset(tensor_item_ids, tensor_labels)
    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

    # 2) Model / optim
    device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    model = BertModel(
        vocab_size=len(unique_item_id) + 1,
        hidden_size=256,
        intermediate_size=256 * 12,
        num_heads=4,
        num_layers=2,
        max_len=max_len
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

    # Track best + history
    best = {
        "epoch": -1,
        "HR": -1.0,
        "Precision": 0.0,
        "Recall": 0.0,
        "MRR": 0.0,
        "NDCG": 0.0,
        "Coverage": 0.0,
    }
    history = []

    # 3) Train loop
    for epoch in range(epoch_num):
        model.train()
        epoch_loss = 0.0

        for item_ids, labels in tqdm(train_dataloader, desc=f"Fold {fold_num} Epoch {epoch+1}", unit="batch"):
            item_ids, labels = item_ids.to(device), labels.to(device)
            key_padding_mask = (item_ids == 0)

            optimizer.zero_grad()
            outputs = model(item_ids, key_padding_mask=key_padding_mask)  # [B, T, V]
            loss = criterion(outputs.view(-1, len(unique_item_id) + 1), labels.view(-1))
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.item())

        print(f"Epoch {epoch+1}, Loss {epoch_loss:.4f}")

        # 4) Evaluate
        model.eval()
        val_item_sequences = {u: seq for u, seq in user_item_to_index_sequence.items() if u in val_users}
        with torch.no_grad():
            val_hr, val_precision_at_k, val_recall_at_k, val_mrr, val_ndcg_at_k, val_coverage = evaluate_model(
                model,
                val_item_sequences,
                k=hitrate,
            )

        print(f"Fold {fold_num} Epoch {epoch+1}, Validation HR@{hitrate}: {val_hr:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation Precision@{hitrate}: {val_precision_at_k:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation Recall@{hitrate}: {val_recall_at_k:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation MRR: {val_mrr:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation NDCG@{hitrate}: {val_ndcg_at_k:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation Coverage: {val_coverage:.4f}")

        # Log epoch metrics
        history.append({
            "epoch": epoch + 1,
            "loss": epoch_loss,
            "HR": float(val_hr),
            "Precision": float(val_precision_at_k),
            "Recall": float(val_recall_at_k),
            "MRR": float(val_mrr),
            "NDCG": float(val_ndcg_at_k),
            "Coverage": float(val_coverage),
        })

        # 5) Save checkpoint if HR improves
        if val_hr > best["HR"]:
            best.update({
                "epoch": epoch + 1,
                "HR": float(val_hr),
                "Precision": float(val_precision_at_k),
                "Recall": float(val_recall_at_k),
                "MRR": float(val_mrr),
                "NDCG": float(val_ndcg_at_k),
                "Coverage": float(val_coverage),
            })
            save_path = f"models/models_item_with_bert/fold_{fold_num}"
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/best_model.pth")

        # Optional: free cache per epoch
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Return model, best metrics dict, and per-epoch history
    return model, best, history


In [19]:
from sklearn.model_selection import KFold
import os
import torch
import json

set_seed()
kf = KFold(n_splits=5, shuffle=True, random_state=42)
user_list = list(user_item_sequence.keys())

fold_metrics = {}   # fold -> best metrics dict
histories = {}      # optional: per-epoch metric history

for fold_num, (train_idx, val_idx) in enumerate(kf.split(user_list), 1):
    print(f"\nStarting Fold {fold_num}...")
    train_users = [user_list[i] for i in train_idx]
    val_users   = [user_list[i] for i in val_idx]

    # Train; new return signature
    model, best_metrics, history = train_model(
        train_users=train_users,
        val_users=val_users,
        fold_num=fold_num,
        mask_ratio=0.5,
    )

    fold_metrics[fold_num] = best_metrics
    histories[fold_num] = history

    # Save per-fold results
    save_path = f"results/results_item_with_bert/fold_{fold_num}"
    os.makedirs(save_path, exist_ok=True)
    with open(f"{save_path}/results.txt", "w") as f:
        f.write(f"Best Epoch: {best_metrics['epoch']}\n")
        f.write(f"HR@{hitrate}:        {best_metrics['HR']:.4f}\n")
        f.write(f"Precision@{hitrate}: {best_metrics['Precision']:.4f}\n")
        f.write(f"Recall@{hitrate}:    {best_metrics['Recall']:.4f}\n")
        f.write(f"MRR:                  {best_metrics['MRR']:.4f}\n")
        f.write(f"NDCG@{hitrate}:      {best_metrics['NDCG']:.4f}\n")
        f.write(f"Coverage:             {best_metrics['Coverage']:.4f}\n")

    # Optional: dump per-epoch history for plotting/analysis
    with open(f"{save_path}/history.json", "w") as f:
        json.dump(history, f, indent=2)

    # Free memory
    del model
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---- Write overall summary (means across folds) ----
overall_path = "results/results_item_with_bert/overall_results.txt"
os.makedirs(os.path.dirname(overall_path), exist_ok=True)

metric_names = ["HR", "Precision", "Recall", "MRR", "NDCG", "Coverage"]

with open(overall_path, "w") as f:
    for fold in sorted(fold_metrics.keys()):
        m = fold_metrics[fold]
        f.write(
            f"Fold {fold} (Best Epoch {m['epoch']}):\n"
            f"  HR@{hitrate}:        {m['HR']:.4f}\n"
            f"  Precision@{hitrate}: {m['Precision']:.4f}\n"
            f"  Recall@{hitrate}:    {m['Recall']:.4f}\n"
            f"  MRR:                  {m['MRR']:.4f}\n"
            f"  NDCG@{hitrate}:      {m['NDCG']:.4f}\n"
            f"  Coverage:             {m['Coverage']:.4f}\n\n"
        )

    f.write("===========================\n")
    f.write("Mean Metrics Across Folds:\n")
    f.write("===========================\n\n")
    for name in metric_names:
        vals = [fold_metrics[fold][name] for fold in sorted(fold_metrics.keys())]
        mean_val = float(sum(vals) / len(vals)) if vals else 0.0
        f.write(f"Mean {name}: {mean_val:.4f}\n")

print("\n✅ Training finished across all folds:")
for name in metric_names:
    vals = [fold_metrics[f][name] for f in sorted(fold_metrics.keys())]
    mean_val = float(sum(vals) / len(vals)) if vals else 0.0
    print(f"Mean {name}: {mean_val:.4f}")



Starting Fold 1...


Fold 1 Epoch 1: 100%|██████████| 419/419 [00:36<00:00, 11.61batch/s]


Epoch 1, Loss 3404.6466
[Sample 24] GT: 1076484, Pred top-5: [127865, 132738, 1076484, 125465, 172914]
[Sample 75] GT: 136110, Pred top-5: [126335, 127865, 136110, 166633, 123793]
[Sample 81] GT: 174086, Pred top-5: [174086, 126335, 127865, 145906, 137585]
[Sample 116] GT: 152836, Pred top-5: [174086, 126335, 145906, 166633, 152836]
[Sample 126] GT: 123793, Pred top-5: [174086, 126335, 166633, 145906, 123793]
[Sample 136] GT: 131117, Pred top-5: [174086, 126335, 145906, 166633, 131117]
[Sample 152] GT: 136110, Pred top-5: [174086, 126335, 136110, 123793, 132738]
[Sample 191] GT: 174086, Pred top-5: [174086, 166633, 145906, 123793, 137585]
[Sample 203] GT: 136110, Pred top-5: [126335, 127865, 136110, 166633, 145906]
[Sample 225] GT: 1991314, Pred top-5: [2057975, 2557055, 616481, 1991314, 459535]
[Sample 264] GT: 2546911, Pred top-5: [1424883, 1459539, 1982904, 1715008, 2546911]
[Sample 299] GT: 123793, Pred top-5: [174086, 126335, 136110, 145906, 123793]
[Sample 330] GT: 152836, Pred t

Fold 1 Epoch 2: 100%|██████████| 419/419 [00:38<00:00, 10.91batch/s]


Epoch 2, Loss 3267.9287
[Sample 40] GT: 168610, Pred top-5: [136860, 131117, 168610, 123793, 1746190]
[Sample 75] GT: 136110, Pred top-5: [921642, 172027, 450618, 136110, 174086]
[Sample 81] GT: 174086, Pred top-5: [172027, 174086, 125465, 136110, 137585]
[Sample 116] GT: 152836, Pred top-5: [174086, 172027, 152836, 125465, 127865]
[Sample 137] GT: 730008, Pred top-5: [126335, 174086, 127865, 131117, 730008]
[Sample 138] GT: 1547971, Pred top-5: [131533, 123793, 1547971, 1949394, 1378631]
[Sample 140] GT: 1076484, Pred top-5: [921642, 1076484, 2771965, 174086, 172027]
[Sample 152] GT: 136110, Pred top-5: [126335, 174086, 136110, 1076484, 137585]
[Sample 172] GT: 136860, Pred top-5: [126335, 174086, 136860, 131533, 131117]
[Sample 191] GT: 174086, Pred top-5: [174086, 172027, 137585, 131533, 131117]
[Sample 203] GT: 136110, Pred top-5: [174086, 125465, 136110, 168610, 1962198]
[Sample 209] GT: 450618, Pred top-5: [450618, 921642, 125465, 174086, 127865]
[Sample 225] GT: 1991314, Pred to

...

Epoch 8, Loss 3125.5800
[Sample 3] GT: 166633, Pred top-5: [172027, 166633, 137585, 136110, 123793]
[Sample 26] GT: 1950240, Pred top-5: [823534, 683251, 1950240, 2444721, 1788819]
[Sample 77] GT: 1457171, Pred top-5: [1457171, 1459683, 703458, 1586334, 1213427]
[Sample 83] GT: 241461, Pred top-5: [125424, 870184, 126335, 1687082, 241461]
[Sample 89] GT: 1717057, Pred top-5: [1717057, 2885734, 2882239, 699458, 2864831]
[Sample 104] GT: 123793, Pred top-5: [174086, 123793, 130727, 136110, 132738]
[Sample 107] GT: 126335, Pred top-5: [730008, 126335, 166633, 137585, 145906]
[Sample 134] GT: 1819243, Pred top-5: [1744232, 1650003, 1819243, 221704, 1092842]
[Sample 140] GT: 127865, Pred top-5: [921642, 127865, 1687082, 131117, 125465]
[Sample 142] GT: 1191124, Pred top-5: [2859339, 1191124, 1650899, 1955352, 1869056]
[Sample 173] GT: 1968677, Pred top-5: [1798233, 317029, 1968677, 527885, 2596674]
[Sample 180] GT: 174086, Pred top-5: [174086, 166633, 126335, 124553, 145906]
[Sample 199] GT

Fold 5 Epoch 9: 100%|██████████| 419/419 [00:28<00:00, 14.76batch/s]


Epoch 9, Loss 3104.2804
[Sample 3] GT: 166633, Pred top-5: [172027, 123793, 166633, 131533, 132738]
[Sample 4] GT: 1146287, Pred top-5: [1687082, 126335, 1800440, 137585, 1146287]
[Sample 26] GT: 1950240, Pred top-5: [2024903, 1950240, 319191, 2441719, 2845075]
[Sample 30] GT: 351928, Pred top-5: [1428087, 1198944, 351928, 1271853, 2610728]
[Sample 57] GT: 1009845, Pred top-5: [879452, 1479699, 657626, 1840637, 1009845]
[Sample 63] GT: 136110, Pred top-5: [126335, 174086, 145906, 136110, 137585]
[Sample 77] GT: 1457171, Pred top-5: [1457171, 2856326, 627759, 684027, 369899]
[Sample 89] GT: 1717057, Pred top-5: [1539576, 1717057, 2248191, 454564, 818210]
[Sample 97] GT: 1129399, Pred top-5: [1492185, 1764436, 527885, 368421, 1129399]
[Sample 104] GT: 123793, Pred top-5: [123793, 127865, 137585, 136110, 131533]
[Sample 107] GT: 126335, Pred top-5: [126335, 172027, 123793, 174086, 145906]
[Sample 134] GT: 1819243, Pred top-5: [1717957, 1188641, 962489, 2834619, 1819243]
[Sample 140] GT: 1

Fold 5 Epoch 10: 100%|██████████| 419/419 [00:27<00:00, 14.99batch/s]


Epoch 10, Loss 3082.3645
[Sample 3] GT: 166633, Pred top-5: [136110, 137585, 126335, 166633, 145906]
[Sample 63] GT: 136110, Pred top-5: [174086, 136110, 172027, 168610, 125465]
[Sample 77] GT: 1457171, Pred top-5: [1949394, 1698166, 1378631, 1457171, 450618]
[Sample 88] GT: 786827, Pred top-5: [1186923, 1626903, 786827, 125424, 1366101]
[Sample 97] GT: 1129399, Pred top-5: [1738544, 1889597, 323450, 1129399, 417055]
[Sample 104] GT: 123793, Pred top-5: [136110, 123793, 126335, 172027, 124553]
[Sample 107] GT: 126335, Pred top-5: [136110, 126335, 174086, 123793, 137585]
[Sample 134] GT: 1819243, Pred top-5: [877767, 657626, 1252187, 1819243, 134393]
[Sample 140] GT: 127865, Pred top-5: [124553, 1076484, 125465, 127865, 1334728]
[Sample 160] GT: 1994009, Pred top-5: [1375928, 1769671, 1539576, 823534, 1994009]
[Sample 174] GT: 1390540, Pred top-5: [1308832, 2955585, 1390540, 1904669, 2215751]
[Sample 180] GT: 174086, Pred top-5: [137585, 174086, 123793, 172027, 126335]
[Sample 212] GT: 